# Meta-NATH CAD Kaggle Workflow

Thin Kaggle orchestration notebook for branch `taitrn`.

The real logic stays in `.py` files. This notebook only clones the repo, links datasets, runs the verified Phase 3.0 conservative workflow, prints acceptance, and packages artifacts.

## Before Running

- Kaggle accelerator: GPU.
- Internet: on, unless the repo/model cache is mounted as Kaggle inputs.
- MVTec input should contain category folders like `bottle`, `cable`, `capsule`, ... either directly or one folder down.
- DINOv2 is intentional for this project stage. DINOv3 is not used here.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/taitrn/NestedLearningForCAD.git"
BRANCH = "taitrn"
REPO = Path("/kaggle/working/NestedLearningForCAD")

# Change these if your Kaggle input names differ.
MVTEC_INPUT = Path(os.environ.get("MVTEC_INPUT", "/kaggle/input/mvtec-ad"))
VISA_INPUT = Path(os.environ.get("VISA_INPUT", "/kaggle/input/visa-ad"))

# Default verified path: Phase 3.0 conservative 8-task.
MAX_TASKS = int(os.environ.get("MAX_TASKS", "8"))
RUN_TESTS = os.environ.get("RUN_TESTS", "1")
RUN_SCORE_COMPARE = os.environ.get("RUN_SCORE_COMPARE", "1")

# Optional heavier runs. Turn on only after the 8-task gate passes.
RUN_PHASE3_15 = False
RUN_VISA = False
INSTALL_DEPS = False
INCLUDE_CHECKPOINTS_IN_ZIP = False

def run(cmd, cwd=None, env=None):
    print("\n$", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=env, check=True)

print("repo:", REPO)
print("branch:", BRANCH)
print("mvtec_input:", MVTEC_INPUT)
print("max_tasks:", MAX_TASKS)

## Clone Or Update Repo

In [ ]:
if REPO.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=REPO)
    run(["git", "checkout", BRANCH], cwd=REPO)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO)
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO])

print("Repo ready:", REPO)

## Install Dependencies And Link Dataset

In [ ]:
if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO)
else:
    print("Skipping pip install. Set INSTALL_DEPS=True if Kaggle image lacks dependencies.")

MVTEC_CATS = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut"]
VISA_CATS = ["candle", "capsules", "cashew", "chewinggum", "fryum", "macaroni1", "macaroni2", "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum"]

def has_categories(path, cats):
    return path.exists() and all((path / cat).is_dir() for cat in cats[:3])

def find_dataset_root(preferred, cats):
    candidates = []
    if preferred.exists():
        candidates.append(preferred)
        candidates.extend([p for p in preferred.iterdir() if p.is_dir()])
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for ds in input_root.iterdir():
            if ds.is_dir():
                candidates.append(ds)
                candidates.extend([p for p in ds.iterdir() if p.is_dir()])
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if has_categories(candidate, cats):
            return candidate
    raise FileNotFoundError(f"Could not find dataset root under /kaggle/input with categories: {cats[:3]}")

def link_dataset(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        print("Dataset target already exists:", dst)
        return
    try:
        os.symlink(src, dst, target_is_directory=True)
        print("Symlinked", src, "->", dst)
    except OSError:
        print("Symlink failed; copying dataset. This may take a while.")
        shutil.copytree(src, dst)

mvtec_root = find_dataset_root(MVTEC_INPUT, MVTEC_CATS)
link_dataset(mvtec_root, REPO / "data" / "mvtec")
print("MVTec root:", mvtec_root)

## Sanity Check

In [ ]:
print("Python:", sys.executable)
run([sys.executable, "-m", "py_compile", "training/run_experiment.py", "training/consolidation_engine.py"], cwd=REPO)
run([sys.executable, "-m", "py_compile", "scripts/run_phase3_consolidation.py", "scripts/evaluate_checkpoint.py", "scripts/phase3_acceptance.py", "scripts/compare_checkpoint_scores.py"], cwd=REPO)
run(["bash", "-n", "scripts/run_server_phase3.sh"], cwd=REPO)
run(["bash", "-n", "scripts/run_server_visa.sh"], cwd=REPO)
if RUN_TESTS == "1":
    run([sys.executable, "scripts/test_integration_2.py"], cwd=REPO)
else:
    print("Skipping integration tests.")

## Run Verified Phase 3.0 Conservative Workflow

This calls `scripts/run_server_phase3.sh` with `MAX_TASKS=8` by default. It performs anchor warmup, before eval, conservative consolidation, after eval, and metric-gated acceptance.

In [ ]:
env = os.environ.copy()
env.update({
    "PYTHON_BIN": sys.executable,
    "MAX_TASKS": str(MAX_TASKS),
    "RUN_TESTS": "0",  # already handled above
    "RUN_SCORE_COMPARE": RUN_SCORE_COMPARE,
})
run(["bash", "scripts/run_server_phase3.sh"], cwd=REPO, env=env)

## Inspect Acceptance Report

In [ ]:
def latest_file(pattern):
    files = sorted(REPO.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(pattern)
    return files[-1]

acceptance_path = latest_file("results/MetaNATH_Phase3_*/acceptance_report.json")
phase3_summary_path = acceptance_path.parent / "phase3_summary.json"
print("acceptance:", acceptance_path)
print(json.dumps(json.loads(acceptance_path.read_text()), indent=2))
print("\nphase3 summary:", phase3_summary_path)
print(json.dumps(json.loads(phase3_summary_path.read_text()), indent=2)[:4000])

## Optional: Try Phase 3.0 On 15 Tasks

Run this only after the 8-task acceptance gate passes. It can take much longer because `store_images=true` is needed for Phase 3 anchors.

In [ ]:
if RUN_PHASE3_15:
    env15 = os.environ.copy()
    env15.update({
        "PYTHON_BIN": sys.executable,
        "MAX_TASKS": "15",
        "RUN_TESTS": "0",
        "RUN_SCORE_COMPARE": "0",
    })
    run(["bash", "scripts/run_server_phase3.sh"], cwd=REPO, env=env15)
else:
    print("Skipping Phase 3 15-task run. Set RUN_PHASE3_15=True after 8-task passes.")

## Optional: VisA Phase 1-2

Use only if a VisA Kaggle input is mounted and contains the 12 category folders expected by `conf/config_visa.yaml`.

In [ ]:
if RUN_VISA:
    visa_root = find_dataset_root(VISA_INPUT, VISA_CATS)
    link_dataset(visa_root, REPO / "data" / "visa")
    visa_env = os.environ.copy()
    visa_env.update({"PYTHON_BIN": sys.executable})
    run(["bash", "scripts/run_server_visa.sh"], cwd=REPO, env=visa_env)
else:
    print("Skipping VisA. Set RUN_VISA=True and mount VisA input when ready.")

## Package Artifacts

The zip includes JSON/YAML/log/text/markdown artifacts. Checkpoints are excluded by default to keep the output small.

In [ ]:
artifact_zip = Path("/kaggle/working/metanath_kaggle_artifacts.zip")
allowed_suffixes = {".json", ".yaml", ".yml", ".log", ".txt", ".md", ".csv"}
if INCLUDE_CHECKPOINTS_IN_ZIP:
    allowed_suffixes.update({".pt", ".pth", ".ckpt"})

with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root_name in ["results", "logs"]:
        root = REPO / root_name
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in allowed_suffixes:
                zf.write(path, path.relative_to(REPO))

print("artifact_zip:", artifact_zip)
print("size_mb:", round(artifact_zip.stat().st_size / (1024 * 1024), 2))